# Fresh?thawed meat classification: domain adaptation experiments

Ten numbered tables provide a stable index for the experiment configuration, supervised references, adaptation results, and H2 ablation.

**Execution is explicit:** `saved` displays a previously calculated run with input/result hash checks; `recompute` trains and evaluates every method over 25 splits. Tables 1, 3, and 4 describe configuration. The other tables contain calculated experimental results.

## Table index

1. [Table 1: RSDE model configuration](#table-01)
2. [Table 2: In-domain DUPLEX holdout: accuracy and macro-F1](#table-02)
3. [Table 3: Domain adaptation parameter settings](#table-03)
4. [Table 4: Hybrid H1 and H2 configurations](#table-04)
5. [Table 5: Source-only and supervised reference performance](#table-05)
6. [Table 6: Domain adaptation performance on OM](#table-06)
7. [Table 7: Domain adaptation performance on TP](#table-07)
8. [Table 8: Domain adaptation performance on TB](#table-08)
9. [Table 9: H2 ablation: normalized performance-versus-fraction AUC](#table-09)
10. [Table 10: Paired H2 AUC differences and corrected confidence intervals](#table-10)

In [1]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "github_ready"]
ROOT = next((p for p in candidates if (p / "run_experiments.py").is_file()
             and (p / "src" / "models.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the experiment repository or its notebooks folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import run_experiments as E
from IPython.display import display

DATA = ROOT / "data" / "Data_Parastar_corrected.csv"
display(E.pd.DataFrame(E.VERSIONS.items(), columns=["Software", "Version"]))

,Software,Version
0,python,3.10.18
1,numpy,2.2.6
2,pandas,2.3.3
3,scipy,1.15.2
4,scikit-learn,1.7.2
5,joblib,1.5.3


## Execution settings

- `MODE = "saved"`: load the included calculated results and regenerate summaries; model fitting is skipped.
- `MODE = "recompute"`: rerun all 25 splits, including TCA, SNV+TCA, and H1. This is substantially slower.
- `RESUME = True`: continue compatible checkpoints in recompute mode.
- `JOBS`: number of worker processes, with one native numerical thread per process.

A new computation writes to `results/local_run/`. The included run is in `results/example_run/`.

In [2]:
MODE = "saved"  # Change to "recompute" to rerun the models.
JOBS = 2
RESUME = False
OUT = ROOT / "results" / ("example_run" if MODE == "saved" else "local_run")
print("Selected mode:", MODE)
print("Result directory:", OUT.relative_to(ROOT))

Selected mode: saved
Result directory: results\example_run


## Dataset and split checks

Raw spectra are used. OM is the labelled source; TP and TB are adaptation targets, with OM included as a same-domain check. The repeated protocol splits biological fillet IDs 70/30 with seeds 0?24. Replicate scans remain together. Dataset attribution is in `data/README.md`.

In [3]:
df, X, sid, scan_type, y, xcols = E.R.load_data(DATA)
E.R.validate_dataset(df, X, sid, xcols)
print(f"{len(df):,} spectra | {len(E.np.unique(sid))} fillets | {X.shape[1]} features")
print("Dataset SHA256:", E.sha256(DATA))
display(E.R.dataset_accounting(df))
audit = E.R.leakage_audit(sid, scan_type)
print(f"Passed {len(audit)} split/domain isolation and nesting checks.")

4,053 spectra | 153 fillets | 125 features
Dataset SHA256: 63c3fcbf6c3ac92aac040062bd595198464256d23d25a917804f94697bc5bdb3


,Freshness,Scan_type,n_spectra,n_fillets
0,FR,OM,829,153
1,FR,TB,599,118
2,FR,TP,780,153
3,TH,OM,669,129
4,TH,TB,522,100
5,TH,TP,654,128


Passed 75 split/domain isolation and nesting checks.


## Prepare the experimental results

The selected execution mode is printed below. Each table is generated from configuration values or actual calculated scores. The saved run retains full-precision per-split results, holdout predictions, provenance, and partition manifests.

In [4]:
experiment = E.run(mode=MODE, data_path=DATA, outdir=OUT, jobs=JOBS, resume=RESUME)
display(experiment["tracking"])

Execution mode: saved
Loading the included calculated run; models are not retrained in this mode.
Tables 1--10 ready; 10,725 per-split records. Output: results\example_run


,Table,Title,Type,CSV
0,1,RSDE model configuration,Configuration,table01.csv
1,2,In-domain DUPLEX holdout: accuracy and macro-F1,Experiment,table02.csv
2,3,Domain adaptation parameter settings,Configuration,table03.csv
3,4,Hybrid H1 and H2 configurations,Configuration,table04.csv
4,5,Source-only and supervised reference performance,Experiment,table05.csv
5,6,Domain adaptation performance on OM,Experiment,table06.csv
6,7,Domain adaptation performance on TP,Experiment,table07.csv
7,8,Domain adaptation performance on TB,Experiment,table08.csv
8,9,H2 ablation: normalized performance-versus-fra...,Experiment,table09.csv
9,10,Paired H2 AUC differences and corrected confid...,Experiment,table10.csv


<a id="table-01"></a>

## Table 1. RSDE model configuration

Fixed RSDE configuration. Each raw-spectrum estimator uses 60 of 125 wavelengths. After a transform to p dimensions, the subspace size is max(2, round(60p/125)), capped by the available dimension.

**Output:** `table01.csv`. Configuration values; no performance metric.

In [5]:
table = experiment["tables"][1]
display(table)
print("CSV:", (OUT / "table01.csv").relative_to(ROOT))

,Parameter,Value,Role in the model
0,Classifier,RSDE,Random Subspace Discriminant Ensemble
1,Base classifier,LDA-style binary classifier,Uses pooled covariance and class priors
2,n_estimators,30,Number of subspace classifiers constructed
3,max_features,60,Wavelengths randomly selected per subspace
4,Feature sampling,Without replacement,Avoids wavelength repetition within a subspace
5,Probability aggregation,Mean probability,Probabilities averaged across all estimators
6,Covariance regularization,1e-08,Stabilizes covariance-matrix inversion
7,Output,Predicted class,Class with the highest mean probability


CSV: results\example_run\table01.csv


<a id="table-02"></a>

## Table 2. In-domain DUPLEX holdout: accuracy and macro-F1

A separate deterministic DUPLEX holdout is used for this in-domain check. Fresh-spectrum signatures from OM/TP/TB determine the fillet partition. Median imputation is restricted to missing signature blocks for split construction.

**Units:** percent. Scan accuracy, scan macro-F1, and fillet macro-F1 are different metrics and are displayed separately. This table is a single holdout, rather than the 25-split averages in Tables 5?10.

**Output:** `table02.csv`. Detailed predictions: `holdout_predictions.csv`.

In [6]:
table = experiment["tables"][2]
display(table.round(2).rename(columns={
    "accuracy_pct": "Accuracy (%)",
    "scan_macro_f1_pct": "Scan macro-F1 (%)",
    "fillet_macro_f1_pct": "Fillet macro-F1 (%)",
}))
print("CSV:", (OUT / "table02.csv").relative_to(ROOT))

,Mode,n_train_spectra,n_test_spectra,Accuracy (%),Scan macro-F1 (%),Fillet macro-F1 (%)
0,OM,1069,429,86.48,86.08,91.24
1,TP,1022,412,91.02,90.95,95.00
2,TB,800,321,91.90,91.88,96.81


CSV: results\example_run\table02.csv


<a id="table-03"></a>

## Table 3. Domain adaptation parameter settings

Settings are read from the implementation. This experiment uses fixed parameters. No hyperparameter search is executed, and the parameter-selection history is not established by this repository.

**Output:** `table03.csv`. Configuration values.

In [7]:
table = experiment["tables"][3]
display(table)
print("CSV:", (OUT / "table03.csv").relative_to(ROOT))

,Method,Key parameter,Value
0,CORAL,reg,0.001
1,SA,var,0.95
2,TCA,n_comp,50
3,TCA,mu,0.1
4,TCA,gamma,1 / n_features
5,GFK,var,0.95
6,SNV,epsilon,0.0
7,Pseudo-Label,n_iter,2
8,Pseudo-Label,conf_thresh,0.8
9,KMM,B,5.0


CSV: results\example_run\table03.csv


<a id="table-04"></a>

## Table 4. Hybrid H1 and H2 configurations

H1 combines GFK with discriminative TCA. H2 combines source, GFK, and CORAL experts using a source/current-target centroid gate. H2 adaptation and gating use the current unlabeled target subset.

**Output:** `table04.csv`. Configuration and combination mechanisms.

In [8]:
table = experiment["tables"][4]
display(table)
print("CSV:", (OUT / "table04.csv").relative_to(ROOT))

,Hybrid method,Component,Parameter,Value / mechanism
0,H1-GFK-DTCA,GFK + Discriminative TCA,var,0.95
1,H1-GFK-DTCA,Discriminative TCA,n_comp,50
2,H1-GFK-DTCA,Regularization,mu_reg,0.1
3,H1-GFK-DTCA,Discriminative weight,lambda (alpha),1.0
4,H2-Ensemble,Source-only expert,RSDE,OM model without transformation
5,H2-Ensemble,GFK expert,GFK + RSDE,Uses the current unlabeled target subset
6,H2-Ensemble,CORAL expert,CORAL + RSDE,Uses the current unlabeled target subset
7,H2-Ensemble,Target information,Current target only,No access to the alternate packaged domain
8,H2-Ensemble,Gating,Source/target centroid distance,Centroids from source + current unlabeled targ...
9,H2-Ensemble,Gating scale (sigma),Median positive squared distance,Computed from source/current-target adaptation...


CSV: results\example_run\table04.csv


<a id="table-05"></a>

## Table 5. Source-only and supervised reference performance

Three reference conditions share the same outer partitions: source-only OM training, in-domain supervised training, and supervised pooled OM/current-domain training. Supervised target labels are used only for the explicitly supervised references.

**Metric:** fillet?condition macro-F1 (%), mean [corrected 95% interval] over 25 splits. Replicate probabilities are averaged within each fillet?condition unit.

**Output:** `table05.csv`. Full precision: `references.csv`.

In [9]:
table = experiment["tables"][5]
display(table)
print("CSV:", (OUT / "table05.csv").relative_to(ROOT))

,Scenario,OM,TP,TB
0,Source-only (OM -> target),"88.0 [82.7, 93.4]","40.5 [32.4, 48.6]","60.4 [50.7, 70.1]"
1,In-domain supervised,"88.0 [82.7, 93.4]","93.5 [89.9, 97.1]","96.0 [92.7, 99.3]"
2,Supervised pooled-domain reference,"88.0 [82.7, 93.4]","85.9 [80.7, 91.1]","91.0 [86.1, 95.9]"


CSV: results\example_run\table05.csv


<a id="table-06"></a>

## Table 6. Domain adaptation performance on OM

OM is the same-domain check. Columns are nested unlabeled target-training fractions, selected by fillet. The 0% source-only point is retained in raw results and in the AUC calculation.

**Metric:** fillet?condition macro-F1 (%), mean [corrected 95% interval], 25 splits.

**Output:** `table06.csv`. Full precision: `summary_f1.csv`.

In [10]:
table = experiment["tables"][6]
display(table)
print("CSV:", (OUT / "table06.csv").relative_to(ROOT))

,Method,5%,10%,20%,50%,80%,100%
0,CORAL,"85.7 [80.2, 91.3]","86.8 [81.6, 91.9]","87.3 [82.3, 92.2]","87.9 [83.1, 92.7]","87.9 [82.8, 93.0]","88.0 [82.7, 93.4]"
1,SA,"62.7 [52.8, 72.7]","62.7 [55.0, 70.4]","64.8 [58.4, 71.2]","64.9 [59.2, 70.5]","65.0 [58.5, 71.4]","64.9 [58.8, 70.9]"
2,TCA,"74.6 [66.7, 82.6]","74.6 [67.2, 82.1]","75.0 [66.9, 83.1]","75.6 [67.2, 84.0]","74.8 [66.7, 82.9]","74.7 [67.3, 82.2]"
3,GFK,"88.1 [83.7, 92.5]","87.5 [82.7, 92.3]","87.7 [83.1, 92.3]","87.9 [82.5, 93.4]","87.5 [82.6, 92.4]","87.6 [82.5, 92.7]"
4,H1-GFK-DTCA,"72.9 [65.6, 80.2]","72.7 [61.9, 83.5]","71.8 [63.6, 80.0]","73.2 [64.8, 81.6]","73.8 [66.5, 81.2]","74.1 [65.0, 83.1]"
5,H2,"88.2 [83.5, 92.9]","87.8 [82.6, 93.0]","88.0 [83.0, 92.9]","88.2 [83.3, 93.1]","88.2 [83.2, 93.3]","88.1 [82.7, 93.5]"
6,SNV+CORAL,"83.5 [77.9, 89.1]","85.1 [81.3, 88.8]","85.4 [80.2, 90.6]","86.7 [81.9, 91.5]","87.2 [82.0, 92.3]","87.1 [82.0, 92.2]"
7,SNV+SA,"49.2 [33.8, 64.6]","46.2 [33.1, 59.3]","44.2 [34.0, 54.4]","43.4 [33.5, 53.4]","43.5 [32.8, 54.2]","42.6 [32.6, 52.6]"
8,SNV+TCA,"76.2 [68.5, 83.9]","75.5 [67.5, 83.6]","75.7 [68.3, 83.2]","76.2 [70.1, 82.3]","75.9 [69.4, 82.4]","75.8 [68.1, 83.4]"
9,SNV+GFK,"87.3 [82.2, 92.3]","86.6 [81.6, 91.5]","86.7 [81.9, 91.6]","86.9 [82.6, 91.2]","86.4 [81.2, 91.7]","86.4 [81.2, 91.5]"


CSV: results\example_run\table06.csv


<a id="table-07"></a>

## Table 7. Domain adaptation performance on TP

Transfer from labelled OM to TP. Adaptation uses the selected TP training fillets without their labels; held-out TP fillets are reserved for evaluation.

**Metric:** fillet?condition macro-F1 (%), mean [corrected 95% interval], 25 splits.

**Output:** `table07.csv`. Full precision: `summary_f1.csv`.

In [11]:
table = experiment["tables"][7]
display(table)
print("CSV:", (OUT / "table07.csv").relative_to(ROOT))

,Method,5%,10%,20%,50%,80%,100%
0,CORAL,"53.9 [42.9, 65.0]","54.9 [44.5, 65.3]","56.7 [47.3, 66.0]","56.3 [47.7, 65.0]","56.6 [49.8, 63.5]","56.6 [49.1, 64.0]"
1,SA,"46.5 [41.9, 51.1]","45.9 [40.1, 51.6]","46.3 [41.3, 51.2]","46.7 [41.5, 51.9]","46.7 [41.5, 51.9]","46.5 [41.5, 51.6]"
2,TCA,"49.9 [36.6, 63.2]","47.4 [35.7, 59.1]","47.5 [36.1, 58.9]","47.8 [35.2, 60.3]","52.0 [41.9, 62.1]","52.6 [41.8, 63.3]"
3,GFK,"52.7 [43.0, 62.5]","55.8 [45.8, 65.8]","57.1 [47.1, 67.0]","56.5 [48.5, 64.5]","57.2 [49.8, 64.5]","57.4 [50.0, 64.7]"
4,H1-GFK-DTCA,"50.0 [41.1, 58.9]","50.8 [42.4, 59.2]","50.5 [42.0, 58.9]","48.3 [44.3, 52.2]","47.6 [42.4, 52.8]","46.8 [41.4, 52.1]"
5,H2,"54.6 [45.3, 63.9]","56.5 [48.0, 65.1]","58.7 [49.4, 68.1]","58.2 [50.2, 66.2]","59.2 [52.1, 66.2]","58.7 [50.4, 67.1]"
6,SNV+CORAL,"48.3 [39.8, 56.8]","49.6 [40.6, 58.5]","48.9 [40.5, 57.3]","48.4 [39.9, 56.9]","46.4 [36.6, 56.2]","46.9 [38.8, 55.0]"
7,SNV+SA,"50.4 [41.8, 59.0]","51.1 [43.5, 58.8]","50.0 [43.1, 56.9]","49.9 [41.2, 58.5]","51.0 [43.6, 58.5]","50.6 [43.0, 58.3]"
8,SNV+TCA,"49.1 [37.9, 60.3]","45.9 [37.4, 54.4]","50.8 [39.7, 61.8]","49.6 [38.5, 60.7]","46.4 [39.0, 53.7]","44.3 [35.9, 52.7]"
9,SNV+GFK,"46.7 [40.1, 53.3]","46.4 [40.5, 52.4]","46.8 [40.9, 52.8]","46.9 [42.6, 51.2]","46.8 [42.3, 51.3]","47.6 [42.2, 52.9]"


CSV: results\example_run\table07.csv


<a id="table-08"></a>

## Table 8. Domain adaptation performance on TB

Transfer from labelled OM to TB. Evaluation uses held-out fillets with available TB measurements. Missing measurements are not imputed for model fitting or evaluation.

**Metric:** fillet?condition macro-F1 (%), mean [corrected 95% interval], 25 splits.

**Output:** `table08.csv`. Full precision: `summary_f1.csv`.

In [12]:
table = experiment["tables"][8]
display(table)
print("CSV:", (OUT / "table08.csv").relative_to(ROOT))

,Method,5%,10%,20%,50%,80%,100%
0,CORAL,"63.3 [49.9, 76.7]","68.2 [56.3, 80.2]","69.8 [62.4, 77.1]","69.8 [62.1, 77.5]","69.9 [62.8, 77.0]","70.5 [64.4, 76.7]"
1,SA,"61.0 [48.7, 73.3]","62.6 [52.2, 73.0]","62.4 [51.8, 73.1]","62.8 [54.7, 71.0]","62.4 [53.1, 71.7]","62.5 [53.7, 71.4]"
2,TCA,"61.8 [49.9, 73.7]","57.6 [44.1, 71.1]","54.9 [42.6, 67.1]","48.1 [35.2, 61.0]","57.1 [46.4, 67.9]","60.0 [48.3, 71.7]"
3,GFK,"65.8 [54.3, 77.2]","69.8 [58.4, 81.2]","70.3 [62.3, 78.3]","70.6 [63.5, 77.8]","70.4 [64.5, 76.3]","69.5 [64.8, 74.3]"
4,H1-GFK-DTCA,"64.6 [55.3, 73.8]","66.0 [58.0, 74.0]","64.2 [53.9, 74.5]","62.2 [51.6, 72.8]","62.4 [51.8, 73.0]","61.4 [50.1, 72.6]"
5,H2,"66.9 [56.1, 77.7]","70.1 [60.3, 80.0]","70.8 [63.4, 78.3]","71.7 [64.7, 78.6]","71.9 [65.0, 78.7]","72.4 [65.3, 79.5]"
6,SNV+CORAL,"62.9 [53.0, 72.7]","63.7 [56.2, 71.2]","64.0 [56.1, 71.9]","63.3 [55.1, 71.6]","63.6 [55.9, 71.4]","62.3 [53.0, 71.7]"
7,SNV+SA,"48.6 [38.5, 58.6]","46.2 [33.9, 58.5]","48.2 [37.4, 59.0]","47.3 [34.6, 60.1]","48.1 [37.0, 59.3]","49.2 [39.2, 59.2]"
8,SNV+TCA,"60.3 [48.9, 71.6]","60.8 [49.3, 72.3]","60.5 [49.6, 71.5]","59.7 [46.6, 72.9]","56.2 [41.3, 71.0]","57.9 [43.8, 72.0]"
9,SNV+GFK,"65.8 [54.9, 76.8]","67.9 [57.9, 78.0]","70.1 [62.7, 77.6]","70.6 [65.3, 75.9]","70.9 [64.6, 77.2]","70.5 [64.3, 76.7]"


CSV: results\example_run\table08.csv


<a id="table-09"></a>

## Table 9. H2 ablation: normalized performance-versus-fraction AUC

The three experts and aggregation rules use the same split and current target subset. AUC integrates macro-F1 over target fractions 0%, 5%, 10%, 20%, 50%, 80%, and 100%, normalized by the fraction range.

**Units:** 0?1. This quantity describes performance against target-data availability; it is not ROC-AUC. Values are averaged across 25 split-specific curves.

**Output:** `table09.csv`. Per-split AUC: `ablation_auc_per_split.csv`.

In [13]:
table = experiment["tables"][9]
display(table.assign(**{
    "TP AUC": table["TP AUC"].map(lambda value: f"{value:.4f}"),
    "TB AUC": table["TB AUC"].map(lambda value: f"{value:.4f}"),
}))
print("CSV:", (OUT / "table09.csv").relative_to(ROOT))

mode,Aggregation / expert,TP AUC,TB AUC
0,Source-only expert,0.4050,0.6040
1,GFK expert,0.5622,0.6982
2,CORAL expert,0.5587,0.6920
3,Equal probability average,0.5408,0.6953
4,Majority vote,0.5660,0.6910
5,Confidence weighting,0.5102,0.6941
6,H2,0.5785,0.7099


CSV: results\example_run\table09.csv


<a id="table-10"></a>

## Table 10. Paired H2 AUC differences and corrected confidence intervals

Differences are calculated within each of the 25 paired split-specific AUCs. Confidence intervals use the corrected repeated-resampling variance with factor 1/25 + 0.30/0.70. Holm-adjusted p-values are included in the detailed statistics.

**Metric:** H2-minus-comparator AUC difference [corrected 95% interval].

**Output:** `table10.csv`. Full precision and p-values: `ablation_paired_statistics_full_precision.csv`.

In [14]:
table = experiment["tables"][10]
display(table)
print("CSV:", (OUT / "table10.csv").relative_to(ROOT))
paired_statistics = E.pd.read_csv(OUT / "ablation_paired_statistics_full_precision.csv")
display(paired_statistics[["mode", "comparison", "mean_diff_auc", "ci95_lo", "ci95_hi", "p_raw", "p_holm"]])

,Comparison,TP delta AUC [95% CI],TB delta AUC [95% CI]
0,H2 - Equal probability average,"0.0377 [-0.0298, 0.1053]","0.0146 [-0.0109, 0.0402]"
1,H2 - Majority vote,"0.0125 [-0.0212, 0.0462]","0.0189 [-0.0350, 0.0729]"
2,H2 - Confidence weighting,"0.0683 [-0.0112, 0.1478]","0.0158 [-0.0222, 0.0539]"


CSV: results\example_run\table10.csv


,mode,comparison,mean_diff_auc,ci95_lo,ci95_hi,p_raw,p_holm
0,TP,H2 - Equal probability average,0.037718,-0.029817,0.105253,0.260395,0.520790
1,TP,H2 - Majority vote,0.012475,-0.021218,0.046167,0.452224,0.520790
2,TP,H2 - Confidence weighting,0.068300,-0.011248,0.147848,0.089080,0.267239
3,TB,H2 - Equal probability average,0.014627,-0.010949,0.040204,0.249432,0.748297
4,TB,H2 - Majority vote,0.018917,-0.035041,0.072874,0.476321,0.798079
5,TB,H2 - Confidence weighting,0.015849,-0.022248,0.053946,0.399040,0.798079


## Result traceability

The ten CSV tables are indexed by `table_index.csv`. The full run contains 10,725 per-split evaluation records. Scores retain full precision before display rounding. `provenance.json` records hashes, input data identity, software versions, and whether the run was calculated previously or recomputed.

In [15]:
print("Execution mode:", experiment["mode"])
print("Per-split records:", len(experiment["full"]))
print("Table index:", (OUT / "table_index.csv").relative_to(ROOT))
print("Provenance:", (OUT / "provenance.json").relative_to(ROOT))

Execution mode: saved
Per-split records: 10725
Table index: results\example_run\table_index.csv
Provenance: results\example_run\provenance.json
